# AVR-2: Анализ временных рядов (итоговое задание)
**Данные:** суточные минимальные температуры, Мельбурн, 1981-1990 (BOM Australia, зеркало: https://github.com/jbrownlee/Datasets, файл `daily-min-temperatures.csv`, копия в `data/`).
**Горизонт прогноза:** h = 90 дней. **Holdout:** последние 90 дней ряда (03.10.1990 - 31.12.1990). Остальное - train.

In [1]:
import os, time, warnings
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
os.makedirs("img", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("data", exist_ok=True)
np.random.seed(42)
print("ok")

ok


## Задача 1. Загрузка данных и EDA

In [2]:
raw = pd.read_csv("data/daily-min-temperatures.csv")
raw.columns = ["ds", "y"]
raw["ds"] = pd.to_datetime(raw["ds"].str.strip())
print("строк в файле:", len(raw), "| пропусков:", int(raw["y"].isna().sum()),
      "| дубликатов дат:", int(raw["ds"].duplicated().sum()))
full_idx = pd.date_range(raw["ds"].min(), raw["ds"].max(), freq="D")
df = pd.DataFrame({"ds": full_idx}).merge(raw, on="ds", how="left")
n_gaps = int(df["y"].isna().sum())
print("дней в полном диапазоне:", len(df), "| пропущенных дат:", n_gaps)
print(df[df["y"].isna()])
df["y"] = df["y"].interpolate(method="linear")
df["unique_id"] = "melb"
df[["ds", "y"]].to_csv("data/melbourne_tmin_daily_processed.csv", index=False)
print("обработано и сохранено: data/melbourne_tmin_daily_processed.csv")
print(df["y"].describe().round(2).to_string())
print("период:", df["ds"].min().date(), "-", df["ds"].max().date())

строк в файле: 3650 | пропусков: 0 | дубликатов дат: 0
дней в полном диапазоне: 3652 | пропущенных дат: 2
             ds   y
1460 1984-12-31 NaN
2921 1988-12-31 NaN
обработано и сохранено: data/melbourne_tmin_daily_processed.csv
count    3652.00
mean       11.18
std         4.07
min         0.00
25%         8.30
50%        11.00
75%        14.00
max        26.30
период: 1981-01-01 - 1990-12-31


In [3]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
stat, pval, *_ = adfuller(df["y"].values)[:2]
print(f"ADF: статистика = {stat:.2f}, p-value = {pval:.5f} -> ряд стационарен в уровнях (d=0 обоснован)")
tmp = df.set_index("ds")["y"]
monthly = tmp.groupby(tmp.index.month).mean()
print("среднее по месяцам:", monthly.round(1).to_dict())
decomp = seasonal_decompose(tmp, model="additive", period=365, extrapolate_trend="freq")
var_tot = tmp.var()
print(f"доля дисперсии: тренд {decomp.trend.var()/var_tot:.2f}, "
      f"сезонность {decomp.seasonal.var()/var_tot:.2f}, остаток {decomp.resid.var()/var_tot:.2f}")

ADF: статистика = -4.44, p-value = 0.00025 -> ряд стационарен в уровнях (d=0 обоснован)
среднее по месяцам: {1: 15.0, 2: 15.4, 3: 14.6, 4: 12.1, 5: 9.9, 6: 7.3, 7: 6.7, 8: 7.9, 9: 9.0, 10: 10.3, 11: 12.5, 12: 13.9}
доля дисперсии: тренд 0.01, сезонность 0.58, остаток 0.40


In [4]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df["ds"], df["y"], lw=0.7)
ax.set_title("Суточный минимум температуры, Мельбурн 1981-1990")
ax.set_xlabel("дата"); ax.set_ylabel("t, C")
fig.tight_layout(); fig.savefig("img/01_series.png"); plt.close(fig)
print("img/01_series.png")

img/01_series.png


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
tmp = df.set_index("ds")["y"]
tmp.groupby(tmp.index.month).apply(list)
data_m = [tmp[tmp.index.month == m].values for m in range(1, 13)]
axes[0].boxplot(data_m, tick_labels=list(range(1, 13)))
axes[0].set_title("Распределение по месяцам (годовая сезонность)")
axes[0].set_xlabel("месяц"); axes[0].set_ylabel("t, C")
tmp.groupby(tmp.index.year).mean().plot(ax=axes[1], marker="o")
axes[1].set_title("Среднегодовая температура")
axes[1].set_xlabel("год"); axes[1].set_ylabel("t, C")
fig.tight_layout(); fig.savefig("img/02_seasonality.png"); plt.close(fig)
print("img/02_seasonality.png")

img/02_seasonality.png


In [6]:
fig = decomp.plot()
fig.set_size_inches(12, 8)
fig.suptitle("Декомпозиция (аддитивная, период 365)")
fig.tight_layout(); fig.savefig("img/03_decomposition.png"); plt.close(fig)
print("img/03_decomposition.png")

img/03_decomposition.png


In [7]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
fig, axes = plt.subplots(2, 1, figsize=(12, 7))
plot_acf(df["y"].values, lags=400, ax=axes[0])
axes[0].set_title("ACF (пик на лаге 365 - годовая сезонность)")
plot_pacf(df["y"].values, lags=30, ax=axes[1])
axes[1].set_title("PACF (лаг 1 доминирует, значимы до лага ~7)")
fig.tight_layout(); fig.savefig("img/04_acf_pacf.png"); plt.close(fig)
print("img/04_acf_pacf.png")

img/04_acf_pacf.png


## Постановка задачи
Прогноз h = 90 дней вперед. Holdout - последние 90 дней (03.10.1990-31.12.1990, n=3562 train / 90 test).
Метрики: RMSE, MAE, MAPE (нули в ряде - 1 значение 0.0 - из MAPE исключены, знаменатель < 0.5 C исключен), sMAPE.
Дополнительно: rolling-бектест (3 окна по 90 дней), интервалы 80/95, анализ остатков лучшей модели.

In [8]:
H = 90
train = df.iloc[:-H].copy()
test = df.iloc[-H:].copy()
print("train:", len(train), train["ds"].min().date(), "-", train["ds"].max().date())
print("test: ", len(test), test["ds"].min().date(), "-", test["ds"].max().date())

def rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))
def mae(a, b): return float(np.mean(np.abs(a - b)))
def mape(a, b):
    m = np.abs(a) >= 0.5
    return float(np.mean(np.abs((a[m] - b[m]) / a[m])) * 100)
def smape(a, b):
    return float(np.mean(2 * np.abs(a - b) / (np.abs(a) + np.abs(b) + 1e-9)) * 100)

def score_row(name, pred):
    y = test["y"].values
    return {"model": name, "RMSE": rmse(y, pred), "MAE": mae(y, pred),
            "MAPE": mape(y, pred), "sMAPE": smape(y, pred)}

train: 3562 1981-01-01 - 1990-10-02
test:  90 1990-10-03 - 1990-12-31


## Задача 2. Статистические методы (statsforecast)
Бейзлайны: Naive, SeasonalNaive(365) - лаг 365 обоснован ACF(365)=0.48.
Ручные: ARIMA(7,0,0) - d=0 по ADF (p<0.001), AR(7) по PACF; HoltWinters AAA m=365 - аддитивные тренд+годовая сезонность; Theta m=365 - классика для сезонных рядов.
Авто: AutoARIMA(m=7), AutoETS(m=365), AutoTheta(m=365).

In [9]:
import time
from statsforecast import StatsForecast
from statsforecast.models import (Naive, SeasonalNaive, ARIMA, AutoARIMA, HoltWinters,
                                  AutoETS, Theta, AutoTheta)
stat_models = [Naive(), SeasonalNaive(season_length=365),
               ARIMA(order=(7, 0, 0)), AutoARIMA(season_length=7),
               HoltWinters(season_length=365), AutoETS(season_length=365, model="ZZZ"),
               Theta(season_length=365), AutoTheta(season_length=365)]
t0 = time.time()
sf = StatsForecast(models=stat_models, freq="D")
fc_stat = sf.forecast(df=train, h=H, level=[80, 95])
t_stat = time.time() - t0
print(f"stat forecast: {t_stat:.1f} c")
stat_names = ["Naive", "SeasonalNaive", "ARIMA", "AutoARIMA", "HoltWinters", "AutoETS", "Theta", "AutoTheta"]
stat_rows = []
for m in stat_names:
    stat_rows.append(score_row(m, fc_stat[m].values))
stat_df = pd.DataFrame(stat_rows).sort_values("RMSE").reset_index(drop=True)
print(stat_df.round(3).to_string(index=False))

stat forecast: 100.1 c
        model  RMSE   MAE   MAPE  sMAPE
  HoltWinters 2.721 2.115 17.168 16.648
        Theta 2.722 2.115 17.109 16.650
    AutoTheta 2.722 2.115 17.109 16.650
    AutoARIMA 3.408 2.688 19.957 21.587
        ARIMA 3.417 2.682 19.887 21.538
        Naive 3.724 2.984 21.943 24.274
SeasonalNaive 3.750 2.891 23.067 23.215
      AutoETS 3.799 3.058 22.397 24.961


## Задача 3а. ML-методы (mlforecast)
Лаги 1-7, 14, 21, 28, 35, 90, 180, 364, 365 (неделя + квартал/полгода/год) + календарные признаки (день недели, месяц, квартал, день года).
Модели: RandomForest (200 деревьев), HistGradientBoosting, Ridge - три разных семейства (бэггинг, бустинг, линейная).

In [10]:
from mlforecast import MLForecast
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
mlf = MLForecast(
    models={"RF": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
            "HGB": HistGradientBoostingRegressor(random_state=42),
            "Ridge": Ridge()},
    freq="D",
    lags=[1, 2, 3, 4, 5, 6, 7, 14, 21, 28, 35, 90, 180, 364, 365],
    date_features=["dayofweek", "month", "quarter", "dayofyear"],
    num_threads=1,
)
t0 = time.time()
mlf.fit(train)
fc_ml = mlf.predict(H)
t_ml = time.time() - t0
print(f"ml fit+predict: {t_ml:.1f} c")
ml_rows = [score_row("ML-" + m, fc_ml[m].values) for m in ["RF", "HGB", "Ridge"]]
ml_df = pd.DataFrame(ml_rows).sort_values("RMSE").reset_index(drop=True)
print(ml_df.round(3).to_string(index=False))

ml fit+predict: 4.7 c
   model  RMSE   MAE   MAPE  sMAPE
  ML-HGB 2.770 2.144 17.599 16.930
ML-Ridge 2.948 2.169 16.311 17.266
   ML-RF 2.986 2.244 17.317 17.950


## Задача 3б. DL-методы (neuralforecast)
input_size=365 (полный годовой цикл), h=90, max_steps=500, random_seed=42.
Архитектуры: LSTM (рекуррентная), MLP (полносвязная), NHITS (иерархическая сверточно-интерполяционная).

In [11]:
from neuralforecast import NeuralForecast
from neuralforecast.models import LSTM, MLP, NHITS
dl_models = [LSTM(h=H, input_size=365, max_steps=500, random_seed=42),
             MLP(h=H, input_size=365, max_steps=500, random_seed=42),
             NHITS(h=H, input_size=365, max_steps=500, random_seed=42)]
nf = NeuralForecast(models=dl_models, freq="D")
t0 = time.time()
nf.fit(df=train)
fc_dl = nf.predict()
t_dl = time.time() - t0
print(f"dl fit+predict: {t_dl:.1f} c")
dl_rows = [score_row("DL-" + m, fc_dl[m].values) for m in ["LSTM", "MLP", "NHITS"]]
dl_df = pd.DataFrame(dl_rows).sort_values("RMSE").reset_index(drop=True)
print(dl_df.round(3).to_string(index=False))

Seed set to 42


Seed set to 42


Seed set to 42


GPU available: True (mps), used: True


TPU available: False, using: 0 TPU cores


...

Predicting: |          | 0/? [00:00<?, ?it/s]

GPU available: True (mps), used: True


TPU available: False, using: 0 TPU cores


Predicting: |          | 0/? [00:00<?, ?it/s]

dl fit+predict: 75.9 c
   model  RMSE   MAE   MAPE  sMAPE
 DL-LSTM 2.669 1.986 16.491 15.814
  DL-MLP 2.879 2.106 16.220 16.757
DL-NHITS 3.054 2.270 17.186 18.389


## Сводная таблица holdout

In [12]:
holdout = pd.concat([stat_df, ml_df, dl_df], ignore_index=True).sort_values("RMSE").reset_index(drop=True)
holdout.to_csv("results/metrics_holdout.csv", index=False)
print(holdout.round(3).to_string(index=False))
best = holdout.iloc[0]
print(f"ЛУЧШАЯ НА HOLDOUT: {best['model']} RMSE={best['RMSE']:.3f} MAE={best['MAE']:.3f} MAPE={best['MAPE']:.2f}%")

        model  RMSE   MAE   MAPE  sMAPE
      DL-LSTM 2.669 1.986 16.491 15.814
  HoltWinters 2.721 2.115 17.168 16.648
        Theta 2.722 2.115 17.109 16.650
    AutoTheta 2.722 2.115 17.109 16.650
       ML-HGB 2.770 2.144 17.599 16.930
       DL-MLP 2.879 2.106 16.220 16.757
     ML-Ridge 2.948 2.169 16.311 17.266
        ML-RF 2.986 2.244 17.317 17.950
     DL-NHITS 3.054 2.270 17.186 18.389
    AutoARIMA 3.408 2.688 19.957 21.587
        ARIMA 3.417 2.682 19.887 21.538
        Naive 3.724 2.984 21.943 24.274
SeasonalNaive 3.750 2.891 23.067 23.215
      AutoETS 3.799 3.058 22.397 24.961
ЛУЧШАЯ НА HOLDOUT: DL-LSTM RMSE=2.669 MAE=1.986 MAPE=16.49%


In [13]:
fam = {"Naive": "Naive", "SeasonalNaive": "SeasonalNaive"}
fam["best_stat"] = stat_df.iloc[0]["model"]
fam["best_ml"] = ml_df.iloc[0]["model"]
fam["best_dl"] = dl_df.iloc[0]["model"]
src = {"Naive": fc_stat, "SeasonalNaive": fc_stat, "best_stat": fc_stat}
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test["ds"].values, test["y"].values, "k-", lw=1.5, label="факт (holdout)")
ax.plot(test["ds"].values, fc_stat["SeasonalNaive"].values, label="SeasonalNaive(365)")
ax.plot(test["ds"].values, fc_stat[fam["best_stat"]].values, label=fam["best_stat"])
ax.plot(fc_ml["ds"].values, fc_ml[fam["best_ml"].replace("ML-", "")].values, label=fam["best_ml"])
ax.plot(fc_dl["ds"].values, fc_dl[fam["best_dl"].replace("DL-", "")].values, label=fam["best_dl"])
ax.set_title("Прогнозы на holdout (90 дней): факт и лучшая модель каждого семейства")
ax.set_xlabel("дата"); ax.set_ylabel("t, C"); ax.legend()
fig.tight_layout(); fig.savefig("img/05_forecasts_test.png"); plt.close(fig)
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(holdout["model"], holdout["RMSE"])
ax.set_title("RMSE на holdout по всем моделям"); ax.tick_params(axis="x", rotation=45)
fig.tight_layout(); fig.savefig("img/06_metrics_bar.png"); plt.close(fig)
print("img/05_forecasts_test.png img/06_metrics_bar.png")

img/05_forecasts_test.png img/06_metrics_bar.png


## Бектестинг: rolling по 3 окна x 90 дней

In [14]:
t0 = time.time()
cv_stat = sf.cross_validation(df=df, h=H, n_windows=3, step_size=H, level=[80, 95])
t_cv_stat = time.time() - t0
print(f"stat backtest: {t_cv_stat:.1f} c")
def cv_scores(cv, models):
    out = []
    for m in models:
        g = cv[["cutoff", m]].merge(df[["ds", "y"]], left_on="cutoff", right_on="ds", how="left")
        r = []
        for _, w in cv.groupby("cutoff"):
            yy = df.set_index("ds").loc[w["ds"], "y"].values
            r.append(rmse(yy, w[m].values))
        out.append({"model": m, "RMSE_backtest": float(np.mean(r))})
    return pd.DataFrame(out).sort_values("RMSE_backtest").reset_index(drop=True)
bt_stat = cv_scores(cv_stat, stat_names)
print(bt_stat.round(3).to_string(index=False))

stat backtest: 282.7 c
        model  RMSE_backtest
  HoltWinters          2.664
        Theta          2.726
    AutoTheta          2.726
        ARIMA          3.363
    AutoARIMA          3.423
SeasonalNaive          3.595
      AutoETS          3.731
        Naive          3.915


In [15]:
t0 = time.time()
cv_ml = mlf.cross_validation(df=df, h=H, n_windows=3, step_size=H)
t_cv_ml = time.time() - t0
print(f"ml backtest: {t_cv_ml:.1f} c")
bt_ml = cv_scores(cv_ml, ["RF", "HGB", "Ridge"]).rename(columns={"model": "m"})
bt_ml["model"] = "ML-" + bt_ml["m"]
bt_ml = bt_ml.drop(columns="m")
print(bt_ml.round(3).to_string(index=False))

ml backtest: 10.9 c
 RMSE_backtest    model
         2.582   ML-HGB
         2.660    ML-RF
         2.708 ML-Ridge


In [16]:
t0 = time.time()
cv_dl = nf.cross_validation(df=df, n_windows=3, step_size=H, h=H)
t_cv_dl = time.time() - t0
print(f"dl backtest: {t_cv_dl:.1f} c")
bt_dl = cv_scores(cv_dl, ["LSTM", "MLP", "NHITS"]).rename(columns={"model": "m"})
bt_dl["model"] = "DL-" + bt_dl["m"]
bt_dl = bt_dl.drop(columns="m")
print(bt_dl.round(3).to_string(index=False))
backtest = pd.concat([bt_stat, bt_ml, bt_dl], ignore_index=True).sort_values("RMSE_backtest").reset_index(drop=True)
backtest.to_csv("results/metrics_backtest.csv", index=False)
print(backtest.round(3).to_string(index=False))

GPU available: True (mps), used: True


TPU available: False, using: 0 TPU cores



  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | hist_encoder        | LSTM          | 199 K  | train
7 | mlp_decoder         | MLP           | 16.6 K | train
--------------------------------------------------------------
215 K     Trainable params
0         Non-trainable params
215 K     Total params
0.863     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=500` reached.


Predicting: |          | 0/? [00:00<?, ?it/s]

dl backtest: 77.0 c
 RMSE_backtest    model
         2.599   DL-MLP
         2.882 DL-NHITS
         3.355  DL-LSTM
        model  RMSE_backtest
       ML-HGB          2.582
       DL-MLP          2.599
        ML-RF          2.660
  HoltWinters          2.664
     ML-Ridge          2.708
        Theta          2.726
    AutoTheta          2.726
     DL-NHITS          2.882
      DL-LSTM          3.355
        ARIMA          3.363
    AutoARIMA          3.423
SeasonalNaive          3.595
      AutoETS          3.731
        Naive          3.915


## Вероятностные оценки: покрытие интервалов 80/95 (лучшая статмодель)

In [17]:
best_stat_name = stat_df.iloc[0]["model"]
y_true = test["y"].values
def coverage(model, lev):
    lo = fc_stat[f"{model}-lo-{lev}"].values
    hi = fc_stat[f"{model}-hi-{lev}"].values
    return float(np.mean((y_true >= lo) & (y_true <= hi)))
cov80 = coverage(best_stat_name, 80); cov95 = coverage(best_stat_name, 95)
print(f"{best_stat_name}: покрытие 80% интервала = {cov80:.3f}, 95% интервала = {cov95:.3f}")
pd.DataFrame([{"model": best_stat_name, "cov80": cov80, "cov95": cov95}]).to_csv("results/coverage.csv", index=False)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test["ds"].values, y_true, "k-", lw=1.2, label="факт")
ax.plot(test["ds"].values, fc_stat[best_stat_name].values, label=f"прогноз {best_stat_name}")
ax.fill_between(test["ds"].values, fc_stat[f"{best_stat_name}-lo-95"].values,
                fc_stat[f"{best_stat_name}-hi-95"].values, alpha=0.2, label="интервал 95%")
ax.fill_between(test["ds"].values, fc_stat[f"{best_stat_name}-lo-80"].values,
                fc_stat[f"{best_stat_name}-hi-80"].values, alpha=0.3, label="интервал 80%")
ax.set_title(f"Интервальный прогноз {best_stat_name} на holdout")
ax.set_xlabel("дата"); ax.set_ylabel("t, C"); ax.legend()
fig.tight_layout(); fig.savefig("img/07_intervals.png"); plt.close(fig)
print("img/07_intervals.png")

HoltWinters: покрытие 80% интервала = 0.811, 95% интервала = 0.933
img/07_intervals.png


## Анализ остатков лучшей модели на holdout

In [18]:
from statsmodels.stats.diagnostic import acorr_ljungbox
resid = y_true - fc_stat[best_stat_name].values
lb = acorr_ljungbox(resid, lags=[10], return_df=True)
lb_p = float(lb["lb_pvalue"].iloc[0])
print(f"остатки {best['model']}: среднее={resid.mean():.3f}, std={resid.std():.3f}, Ljung-Box(10) p-value={lb_p:.5f}")
pd.DataFrame([{"model": best["model"], "resid_mean": float(resid.mean()),
               "resid_std": float(resid.std()), "ljungbox_p": lb_p}]).to_csv("results/residuals.csv", index=False)
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(test["ds"].values, resid, lw=0.8)
axes[0].axhline(0, color="k", lw=1); axes[0].set_title(f"Остатки {best['model']} на holdout")
from statsmodels.graphics.tsaplots import plot_acf as _pacf
_pacf(resid, lags=40, ax=axes[1])
axes[1].set_title("ACF остатков")
fig.tight_layout(); fig.savefig("img/08_residuals_acf.png"); plt.close(fig)
print("img/08_residuals_acf.png")

остатки DL-LSTM: среднее=0.067, std=2.720, Ljung-Box(10) p-value=0.00041


img/08_residuals_acf.png

## Выводы по задачам (числа из ячеек выше)

In [19]:
print("=== ЗАДАЧА 1: данные и EDA ===")
print(f"Ряд: {len(df)} дней 1981-1990, пропущенных дат было {n_gaps} (интерполированы). "
      f"Среднее {df['y'].mean():.2f} C, std {df['y'].std():.2f}. ADF p={pval:.5f} - стационарен. "
      "Годовая сезонность: январь ~15 C, июль ~6.7 C.")
print("=== ЗАДАЧА 2: статметоды ===")
print(stat_df.round(3).to_string(index=False))
print("=== ЗАДАЧА 3: ML/DL ===")
print(ml_df.round(3).to_string(index=False))
print(dl_df.round(3).to_string(index=False))
print("=== ЗАДАЧА 4: бектест + интервалы + остатки ===")
print(backtest.round(3).to_string(index=False))
print(f"Покрытие интервалов {best_stat_name}: 80% -> {cov80:.3f}, 95% -> {cov95:.3f}. "
      f"Остатки {best['model']}: Ljung-Box p={lb_p:.5f}.")
print(f"Время: stat {t_stat:.0f}c / ml {t_ml:.0f}c / dl {t_dl:.0f}c / "
      f"бектест stat {t_cv_stat:.0f}c / ml {t_cv_ml:.0f}c / dl {t_cv_dl:.0f}c.")

=== ЗАДАЧА 1: данные и EDA ===
Ряд: 3652 дней 1981-1990, пропущенных дат было 2 (интерполированы). Среднее 11.18 C, std 4.07. ADF p=0.00025 - стационарен. Годовая сезонность: январь ~15 C, июль ~6.7 C.
=== ЗАДАЧА 2: статметоды ===
        model  RMSE   MAE   MAPE  sMAPE
  HoltWinters 2.721 2.115 17.168 16.648
        Theta 2.722 2.115 17.109 16.650
    AutoTheta 2.722 2.115 17.109 16.650
    AutoARIMA 3.408 2.688 19.957 21.587
        ARIMA 3.417 2.682 19.887 21.538
        Naive 3.724 2.984 21.943 24.274
SeasonalNaive 3.750 2.891 23.067 23.215
      AutoETS 3.799 3.058 22.397 24.961
=== ЗАДАЧА 3: ML/DL ===
   model  RMSE   MAE   MAPE  sMAPE
  ML-HGB 2.770 2.144 17.599 16.930
ML-Ridge 2.948 2.169 16.311 17.266
   ML-RF 2.986 2.244 17.317 17.950
   model  RMSE   MAE   MAPE  sMAPE
 DL-LSTM 2.669 1.986 16.491 15.814
  DL-MLP 2.879 2.106 16.220 16.757
DL-NHITS 3.054 2.270 17.186 18.389
=== ЗАДАЧА 4: бектест + интервалы + остатки ===
        model  RMSE_backtest
       ML-HGB          2.582